In [0]:
# ── CONFIG ───────────────────────────────────────────────────────────────────
 
CATALOG        = "clutchlytics"
GOALIE_LOGS    = f"{CATALOG}.silver.nhl_goalie_game_logs"
DIM_ATHLETES   = f"{CATALOG}.silver.dimAthletes"
GOLD_TABLE     = f"{CATALOG}.gold.nhl_gold_goalie_ratings"
 
LEAGUE         = "nhl"
SPORT          = "hockey"
ROUND          = 1       # ← change to 2 when R2 data loads
 
MIN_REG_STARTS    = 10
MIN_R1_STARTS     = 1
MAX_R1_WEIGHT     = 0.6
WEIGHT_PER_START  = 0.2
 
print(f"Source       : {GOALIE_LOGS}")
print(f"Target       : {GOLD_TABLE}")
print(f"Round        : {ROUND}")
print(f"Min reg starts : {MIN_REG_STARTS}")
print(f"Min R1 starts  : {MIN_R1_STARTS}")
print(f"Weight per start: {WEIGHT_PER_START} (caps at {MAX_R1_WEIGHT})")

In [0]:
# ── READ SOURCE ───────────────────────────────────────────────────────────────
 
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import datetime, timezone
 
goalie_df = spark.table(GOALIE_LOGS)
 
dim_athletes = (
    spark.table(DIM_ATHLETES)
    .filter(F.col("is_current") == True)
    .select(
        F.col("athlete_id").cast("string").alias("dim_athlete_id"),
        F.col("clutch_athlete_id"),
        F.col("clutch_team_id"),
    )
)
 
print(f"Goalie log rows : {goalie_df.count()}")
print(f"dimAthletes     : {dim_athletes.count()}")

In [0]:
# ── REGULAR SEASON BASELINE ───────────────────────────────────────────────────
 
reg_df = (
    goalie_df
    .filter(F.col("season_type") == "regular")
    .groupBy("athlete_id", "athlete_name", "team_abbreviation")
    .agg(
        F.sum(F.col("started_game").cast("int")).alias("reg_games_started"),
        F.sum(F.col("played_in_game").cast("int")).alias("reg_games_played"),
        # Weighted SV% — weight each game by shots_against for accuracy
        F.round(
            F.sum(F.col("saves")) / F.sum(F.col("shots_against")), 3
        ).alias("reg_sv_pct"),
        F.round(
            F.sum(F.col("goals_against")) /
            (F.sum(F.col("toi_seconds")) / 3600.0), 2
        ).alias("reg_gaa"),
        F.sum("shutouts").alias("reg_shutouts"),
        F.sum("wins").alias("reg_wins"),
        F.sum("losses").alias("reg_losses"),
    )
    .filter(F.col("reg_games_started") >= MIN_REG_STARTS)
)
 
print(f"Goalies with reg baseline (>= {MIN_REG_STARTS} starts): {reg_df.count()}")

In [0]:
# ── PLAYOFF PERFORMANCE ───────────────────────────────────────────────────────
 
playoff_df = (
    goalie_df
    .filter(
        (F.col("season_type") == "playoffs") &
        (F.col("round") == ROUND)
    )
    .groupBy("athlete_id")
    .agg(
        F.sum(F.col("started_game").cast("int")).alias("r1_games_started"),
        F.sum(F.col("played_in_game").cast("int")).alias("r1_games_played"),
        # Weighted SV% by shots_against
        F.round(
            F.sum(F.col("saves")) / F.sum(F.col("shots_against")), 3
        ).alias("r1_sv_pct"),
        F.round(
            F.sum(F.col("goals_against")) /
            (F.sum(F.col("toi_seconds")) / 3600.0), 2
        ).alias("r1_gaa"),
        F.sum("shutouts").alias("r1_shutouts"),
        F.sum("wins").alias("r1_wins"),
        F.sum("losses").alias("r1_losses"),
        F.sum("overtime_losses").alias("r1_ot_losses"),
    )
    .filter(F.col("r1_games_started") >= MIN_R1_STARTS)
)
 
print(f"Goalies with R{ROUND} playoff data (>= {MIN_R1_STARTS} starts): {playoff_df.count()}")

In [0]:
# ── JOIN + COMPOSITE RATING ───────────────────────────────────────────────────
 
ratings_df = (
    reg_df
    .join(playoff_df, on="athlete_id", how="inner")
    .join(
        dim_athletes,
        reg_df.athlete_id == dim_athletes.dim_athlete_id,
        how="left"
    )
    .drop("dim_athlete_id")
)
 
# ── Dynamic weighting based on games started ──
# r1_weight  = LEAST(r1_games_started * 0.2, 0.6)
# reg_weight = 1 - r1_weight
# 1 start → 0.2 / 0.8
# 2 starts → 0.4 / 0.6
# 3+ starts → 0.6 / 0.4 (capped)
 
ratings_df = (
    ratings_df
    .withColumn(
        "r1_weight",
        F.least(
            F.col("r1_games_started") * F.lit(WEIGHT_PER_START),
            F.lit(MAX_R1_WEIGHT)
        )
    )
    .withColumn("reg_weight", F.lit(1.0) - F.col("r1_weight"))
)
 
# ── Composite SV% and GAA ──
ratings_df = (
    ratings_df
    .withColumn(
        "composite_sv_pct",
        F.round(
            (F.col("reg_sv_pct") * F.col("reg_weight")) +
            (F.col("r1_sv_pct") * F.col("r1_weight")),
            3
        )
    )
    .withColumn(
        "composite_gaa",
        F.round(
            (F.col("reg_gaa") * F.col("reg_weight")) +
            (F.col("r1_gaa") * F.col("r1_weight")),
            2
        )
    )
)
 
# ── goalie_rating_score — rank-based 0-100 within all active playoff goalies ──
# 100 = best composite SV%, 0 = worst
# Uses PERCENT_RANK for smooth 0-100 distribution
 
rating_window = Window.orderBy(F.col("composite_sv_pct").asc())
 
n_goalies = ratings_df.count()
 
ratings_df = ratings_df.withColumn(
    "goalie_rating_score",
    F.round(
        F.percent_rank().over(rating_window) * 100, 1
    )
)
 
# ── still_active flag ──
# r1_starter — started 3+ games in R1 (primary starter)
ratings_df = (
    ratings_df
    .withColumn("r1_starter", F.col("r1_games_started") >= 3)
    .withColumn(
        "sv_pct_delta",
        F.round(F.col("r1_sv_pct") - F.col("reg_sv_pct"), 3)
    )
    .withColumn(
        "gaa_delta",
        F.round(F.col("r1_gaa") - F.col("reg_gaa"), 2)
    )
)
 
print(f"Goalies in rating model: {ratings_df.count()}")

In [0]:
# ── PREVIEW ───────────────────────────────────────────────────────────────────
 
print(f"── Goalie ratings ranked (Round {ROUND}) ──")
ratings_df.select(
    "athlete_name",
    "team_abbreviation",
    "r1_games_started",
    "r1_weight",
    "reg_sv_pct",
    "r1_sv_pct",
    "composite_sv_pct",
    "composite_gaa",
    "goalie_rating_score",
    "sv_pct_delta",
    "r1_starter",
).orderBy(F.col("goalie_rating_score").desc()).show(30, truncate=False)

In [0]:
# ── FINAL COLUMN ORDER ────────────────────────────────────────────────────────
 
ingested_at = datetime.now(timezone.utc).isoformat()
 
gold_df = ratings_df.select(
    # ── Identity ──
    "athlete_id",
    "athlete_name",
    "clutch_athlete_id",
    "clutch_team_id",
    "team_abbreviation",
 
    # ── Round context ──
    F.lit(ROUND).alias("round"),
    F.lit(2026).alias("season"),
    F.lit(SPORT).alias("sport"),
    F.lit(LEAGUE).alias("league"),
 
    # ── Regular season baseline ──
    "reg_games_started",
    "reg_games_played",
    "reg_sv_pct",
    "reg_gaa",
    "reg_shutouts",
    "reg_wins",
    "reg_losses",
 
    # ── Playoff performance ──
    "r1_games_started",
    "r1_games_played",
    "r1_sv_pct",
    "r1_gaa",
    "r1_shutouts",
    "r1_wins",
    "r1_losses",
    "r1_ot_losses",
 
    # ── Weighting ──
    "r1_weight",
    "reg_weight",
 
    # ── Composite rating ──
    "composite_sv_pct",
    "composite_gaa",
    "goalie_rating_score",
 
    # ── Deltas ──
    "sv_pct_delta",
    "gaa_delta",
 
    # ── Flags ──
    "r1_starter",
 
    # ── Metadata ──
    F.lit(ingested_at).alias("ingested_at"),
    F.lit("silver.nhl_goalie_game_logs").alias("source_table"),
)

In [0]:
# ── WRITE TO GOLD ─────────────────────────────────────────────────────────────
# MERGE on athlete_id + round — R1 rows preserved when R2 is added.
 
table_exists = spark.catalog.tableExists(GOLD_TABLE)
 
if not table_exists:
    (
        gold_df
        .write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(GOLD_TABLE)
    )
    print(f"Table created: {GOLD_TABLE}")
else:
    gold_df.createOrReplaceTempView("new_goalie_ratings")
    spark.sql(f"""
        MERGE INTO {GOLD_TABLE} AS target
        USING new_goalie_ratings AS source
        ON  target.athlete_id = source.athlete_id
        AND target.round      = source.round
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)
    print(f"Merged into existing table: {GOLD_TABLE}")

In [0]:
# ── SANITY CHECKS ─────────────────────────────────────────────────────────────
 
checks = spark.sql(f"""
    SELECT
        COUNT(*)                                                    AS total_goalies,
        COUNT(CASE WHEN r1_starter = true      THEN 1 END)         AS r1_starters,
        COUNT(CASE WHEN r1_weight = 0.6        THEN 1 END)         AS full_weight_goalies,
        COUNT(CASE WHEN r1_weight < 0.6        THEN 1 END)         AS scaled_weight_goalies,
        COUNT(CASE WHEN composite_sv_pct > 1.0 THEN 1 END)         AS bad_sv_pct,
        COUNT(CASE WHEN goalie_rating_score IS NULL THEN 1 END)    AS null_ratings,
        COUNT(CASE WHEN clutch_athlete_id IS NULL  THEN 1 END)     AS null_clutch_ids,
        ROUND(AVG(reg_sv_pct), 3)                                  AS avg_reg_sv_pct,
        ROUND(AVG(r1_sv_pct), 3)                                   AS avg_r1_sv_pct,
        ROUND(AVG(composite_sv_pct), 3)                            AS avg_composite_sv_pct,
        ROUND(AVG(sv_pct_delta), 3)                                AS avg_sv_pct_delta,
        MIN(goalie_rating_score)                                    AS min_score,
        MAX(goalie_rating_score)                                    AS max_score
    FROM {GOLD_TABLE}
    WHERE round = {ROUND}
""")
 
print(f"Sanity checks (Round {ROUND}):")
checks.show(truncate=False)